# Crop Yield Prediction — Random Forest Model

This notebook trains a **Random Forest Regressor** to predict crop yield using the
crop recommendation dataset (`Crop_recommendation.csv`).

### Workflow
1. Load the dataset
2. Explore the data
3. Split into training and test sets
4. Train a Random Forest Regressor
5. Evaluate with R², MAE, and RMSE
6. Plot feature importances
7. Save the trained model

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Libraries loaded successfully.')

## 1. Load the Dataset

The dataset `Crop_recommendation.csv` is located in the same directory as this notebook.
It contains N, P, K, temperature, humidity, ph, rainfall, label, and yield columns.

In [ ]:
# ── Locate the dataset ─────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.getcwd()  # directory where the notebook is running

# Try common file names / extensions for the dataset
_candidates = [
    'Crop_recommendation.csv',
    'crop_recommendation.csv',
    'Crop_recommendation.xlsx',
    'final dataset encoding.csv',
    'final_dataset_encoding.csv',
    'final dataset encoding.xlsx',
    'final_dataset_encoding.xlsx',
]

DATASET_PATH = None
for name in _candidates:
    candidate = os.path.join(NOTEBOOK_DIR, name)
    if os.path.exists(candidate):
        DATASET_PATH = candidate
        break

if DATASET_PATH is None:
    raise FileNotFoundError(
        'Dataset not found. Ensure Crop_recommendation.csv is in the notebooks/ folder.'
    )

print(f'Loading dataset from: {DATASET_PATH}')

In [ ]:
# ── Read the file ──────────────────────────────────────────────────────────────
if DATASET_PATH.endswith('.xlsx'):
    df = pd.read_excel(DATASET_PATH)
else:
    df = pd.read_csv(DATASET_PATH)

print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'None — dataset is complete.')

In [ ]:
# ── Identify the yield (target) column ────────────────────────────────────────
# Common names used in crop-yield datasets
_yield_keywords = ['yield', 'production', 'output', 'hg/ha', 'kg/ha', 'tons']
TARGET_COL = None
for col in df.columns:
    if any(kw in col.lower() for kw in _yield_keywords):
        TARGET_COL = col
        break

if TARGET_COL is None:
    # Fall back: last numeric column
    TARGET_COL = df.select_dtypes(include=[np.number]).columns[-1]
    print(f'[Warning] Could not auto-detect yield column — using last numeric column: "{TARGET_COL}"')
    print('Set TARGET_COL manually if this is incorrect.')
else:
    print(f'Target column detected: "{TARGET_COL}"')

In [ ]:
# ── Yield distribution ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[TARGET_COL], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title(f'Distribution of {TARGET_COL}')
axes[0].set_xlabel(TARGET_COL)
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df[TARGET_COL].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title(f'Boxplot of {TARGET_COL}')
axes[1].set_ylabel(TARGET_COL)

plt.tight_layout()
plt.savefig('yield_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved yield_distribution.png')

## 3. Prepare Features and Target

In [ ]:
# ── Drop rows with missing target ──────────────────────────────────────────────
df_clean = df.dropna(subset=[TARGET_COL]).copy()
print(f'Rows after dropping missing target: {len(df_clean)} (dropped {len(df) - len(df_clean)})')

# ── Feature matrix and target vector ──────────────────────────────────────────
# Keep only numeric columns (dataset is already hot-encoded, so all cols should be numeric)
feature_cols = [c for c in df_clean.columns if c != TARGET_COL]
numeric_feature_cols = df_clean[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

non_numeric = [c for c in feature_cols if c not in numeric_feature_cols]
if non_numeric:
    print(f'[Warning] Dropping non-numeric columns: {non_numeric}')

X = df_clean[numeric_feature_cols].fillna(df_clean[numeric_feature_cols].median())
y = df_clean[TARGET_COL]

print(f'Features : {X.shape[1]} columns, {X.shape[0]} rows')
print(f'Target   : "{TARGET_COL}" — mean={y.mean():.2f}, std={y.std():.2f}')
print('Feature columns:', numeric_feature_cols)

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
)

print(f'Train set : {X_train.shape[0]} samples')
print(f'Test  set : {X_test.shape[0]} samples')

## 5. Train the Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,          # grow until leaves are pure (or min_samples_leaf)
    min_samples_leaf=2,
    max_features='sqrt',     # standard for regression forests
    random_state=RANDOM_SEED,
    n_jobs=-1,               # use all available CPU cores
)

rf_model.fit(X_train, y_train)
print('Random Forest Regressor trained successfully.')

## 6. Evaluate the Model

In [ ]:
y_pred_train = rf_model.predict(X_train)
y_pred_test  = rf_model.predict(X_test)

def print_metrics(y_true, y_pred, label=''):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f'{label:10s}  R² = {r2:.4f}   MAE = {mae:.4f}   RMSE = {rmse:.4f}')
    return r2, mae, rmse

print('=== Model Performance ===')
print_metrics(y_train, y_pred_train, 'Train')
r2, mae, rmse = print_metrics(y_test,  y_pred_test,  'Test')

In [ ]:
# ── 5-fold Cross-Validation on the full training set ──────────────────────────
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
print(f'5-Fold CV R² scores : {cv_scores.round(4)}')
print(f'Mean CV R²          : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ── Actual vs Predicted plot ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred_test, alpha=0.4, color='steelblue', edgecolors='white', linewidths=0.5)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Yield')
ax.set_ylabel('Predicted Yield')
ax.set_title(f'Actual vs Predicted Yield\nTest R² = {r2:.4f}')
ax.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved actual_vs_predicted.png')

## 7. Feature Importances

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=numeric_feature_cols)
importances_sorted = importances.sort_values(ascending=False)

# Show top 30 features (the hot-encoded dataset may have many columns)
top_n = min(30, len(importances_sorted))
top_importances = importances_sorted.head(top_n)

fig, ax = plt.subplots(figsize=(10, max(4, top_n * 0.35)))
top_importances[::-1].plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title(f'Top {top_n} Feature Importances (Random Forest)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved feature_importances.png')

print('\nTop 10 most important features:')
print(importances_sorted.head(10).to_string())

## 8. Save the Trained Model

In [ ]:
MODEL_OUTPUT_PATH = 'crop_yield_rf_model.pkl'
FEATURE_COLS_PATH = 'crop_yield_rf_features.txt'

# Save model
joblib.dump(rf_model, MODEL_OUTPUT_PATH)
print(f'Model saved to: {MODEL_OUTPUT_PATH}')

# Save feature column names so predictions use the same order
with open(FEATURE_COLS_PATH, 'w') as f:
    f.write('\n'.join(numeric_feature_cols))
print(f'Feature list saved to: {FEATURE_COLS_PATH}')

## 9. Example Inference

Load the saved model and run a prediction on a single row to verify everything works end-to-end.

In [ ]:
# ── Reload model ───────────────────────────────────────────────────────────────
loaded_model = joblib.load(MODEL_OUTPUT_PATH)

# Use the first row of the test set as a demo sample
sample = X_test.iloc[[0]]
actual_yield = float(y_test.iloc[0])
predicted_yield = float(loaded_model.predict(sample)[0])

print('=== Example Prediction ===')
print(f'Actual yield    : {actual_yield:.4f}')
print(f'Predicted yield : {predicted_yield:.4f}')
print(f'Error           : {abs(actual_yield - predicted_yield):.4f}')

---
## Summary

| Metric | Value |
|--------|-------|
| **Test R²** | see above |
| **Test MAE** | see above |
| **Test RMSE** | see above |
| **Model** | `crop_yield_rf_model.pkl` |
| **Features** | `crop_yield_rf_features.txt` |

> **To use the model in production** load it with `joblib.load('crop_yield_rf_model.pkl')` and call `.predict(X)` where `X` is a DataFrame with the same columns listed in `crop_yield_rf_features.txt`.